# Objective
We will be building a GNN to suggest similar movies based on user history

**Dataset**


1.   **Rating** Dataset containing following attributes:


*   user_id
*   movie_id
*   rating
*   timestamp


2.   **Movie** dataset to get:

*   movie_id
*   title
*   genre




In [ ]:
#Import the necessary Libraries

import os
from collections import defaultdict
import math
import networkx as nx #To visualize the complex graphs
import random
from tqdm import tqdm #Library used to show Progress bar for loops - to evaluate time consuming tasks
from itertools import combinations
from zipfile import ZipFile
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Dataset Retreival

In [ ]:
urlretrieve(
    "http://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "movielens.zip"
)

ZipFile("movielens.zip", "r").extractall()

# Load the required datasets

In [ ]:
movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")

In [ ]:
max_rating = ratings['rating'].max()
print(f"\nMaximum value of rating column: {max_rating}")


Maximum value of rating column: 5.0


In [ ]:
#Getting Movie Name by Movie ID
def get_movie_title_by_ID(movieId):
  return list(movies[movies['movieId'] == movieId]['title'])[0]

#Get the Movie ID from Name
def get_id_from_title(title):
  return list(movies[movies['title'] == title]['movieId'])[0]


In [ ]:
#Check if a single User had multiple reviews for a Particular Movie
duplicates = (
    ratings.groupby(['userId', 'movieId'])
    .size()
    .reset_index(name='count')
    .query('count > 1')
)
print(duplicates)

Empty DataFrame
Columns: [userId, movieId, count]
Index: []


# Calculate Movie Frequencies

In [ ]:
#Num of Times, movies are rated
movie_frequency = defaultdict(int) #Says how many times each movie is watched

for movieId in ratings['movieId']:
  movie_frequency[movieId] += 1


In [ ]:
#Most Watched Movie

movieId = max(movie_frequency, key=movie_frequency.get)
print(f'',get_movie_title_by_ID(movieId), 'was watched by', movie_frequency[movieId], 'users')

 Forrest Gump (1994) was watched by 329 users


# Calculate Pair Wise Frequency of Movies

In [ ]:
pair_frequency = defaultdict(int)

for user, group in tqdm(ratings.groupby('userId'), desc="Processing Users"):
  movies = group['movieId'].tolist()

  for i,j in combinations(movies, 2):
    #Sort to avoid (10, 20) and (20, 10) counting separately
    pair = tuple(sorted((i,j)))
    pair_frequency[pair] += 1


Processing Users: 100%|██████████| 610/610 [00:54<00:00, 11.21it/s]


In [28]:
for i, (pair, freq) in enumerate(pair_frequency.items()):
  print(f"{pair}: {freq}")
  if i > 10:
    break

(1, 3): 32
(1, 6): 58
(1, 47): 99
(1, 50): 96
(1, 70): 30
(1, 101): 16
(1, 110): 117
(1, 151): 21
(1, 157): 9
(1, 163): 36
(1, 216): 33
(1, 223): 59


If the pair wise frequency of two movies is high, if a new user happens to watch Movie A, they are likely to be suggested Movie B


# Create Grpah


*   **Nodes**: Represent Individual MovieIDs
*   **Edges**: Represent Connection between Movies, where the edge weight indicates the number of times two movies were rated together (pair wise frequency)

To think of this more intutively, the higher the weight of an edge between two movies A and B, the higher is the probability of movie B being suggested after you watched movie A and vice versa

# PMI (Point wise Mutual Information)
The PMI metric is used to measure the strength of assosiation between two items - in this case, two movies

When building a network, using raw **pairwise frequency** as the edge weight will be misleading, because popular movies tends to appear with many others simply due to their popularity, not because of any relationship between movies.

To overcome this, PMI compares the observed co-occurrence probability of two movies with the expected co-occurrence probability if they were independent. This helps identify pairs that co-occur more frequently than would be expected by chance.

There are other measure to calculate the relationship like cosine, jaccard similarity and Pearson correlation coefficient

**Why PMI?**
In co-occurance data, popular items like blockbuster movies - tend to appear frequently with many others, even if they're unrelated
Metric based on raw counts or even jaccardsimilarity are often baised towards popularity

In [57]:
min_weight = 0 #Min weight of edge required to connect two nodes

D = math.log(sum(movie_frequency.values())) #Log(Total Number of Movie Occurances)

#PMI(x,y) = log(P(x,y)/P(x)*P(y)) = log(pair frequency) - log(x frequency) - log(y frequency) + D
movies_graph = nx.Graph() # Undirected graph; use nx.DiGraph() for directed version

#Add weighted edges between movies
for pair, freq in tqdm(pair_frequency.items(), desc="Adding Edges"):
  x,y = pair
  pmi = math.log(freq) - math.log(movie_frequency[x]) - math.log(movie_frequency[y]) + D
  weight = pmi
  if pmi >= min_weight:
    movies_graph.add_edge(pair[0], pair[1], weight=weight)

Adding Edges: 100%|██████████| 13157672/13157672 [01:05<00:00, 201692.94it/s]


In [58]:
print("Total number of graph nodes:", movies_graph.number_of_nodes())
print("Total number of graph edges:", movies_graph.number_of_edges())

Total number of graph nodes: 9724
Total number of graph edges: 13157672


# Calculate Average Degree

The average degree often gives us an idea about inter-connectivity of the nodes.

In [59]:
degrees = []
for node in movies_graph.nodes:
  degrees.append(movies_graph.degree(node))

average_degree = sum(degrees)/len(degrees)
print(f"Average Degree: {average_degree}")

Average Degree: 2706.2262443438913


This gives an idea that on average every node is connected to 287 other nodes

# Create Vocabulary Lookup

In [60]:
vocabulary = ["NA"] + list(movies_graph.nodes)

vocabulary_lookup = {token: idx for idx, token in enumerate(vocabulary)}

# Traverse through the Graph: To pick the neighbour

The below function next_movie() does the simple operation of travelling to next neightbour node given you're currently on a node, i.e., when you watch a movie, what are the next movies you could consider

In [63]:
def next_movie(graph, previous, current, p, q):
  #p, q are probabilites associated with going back to already watched movie, and going to a random movie in the graph which is connected
  neighbors = list(graph.neighbors(current))
  weights = []
  for nei in neighbors:
    weight = graph[current][nei]["weight"]
    if nei == previous:
      #Control the probability to return to Previous Node (As user may like to watch repeated movies)
      weights.append(weight/p)

    elif graph.has_edge(nei, previous):
      #Local movement is possible. So give more prob compared to jumping random in graph
      weights.append(weight)
    else:
      weights.append(weight/q)

  #Compute the Probabilites of visiting the neighbors
  weight_sum = sum(weights)

  prob = [weight/weight_sum for weight in weights]

  #Probabilistic approach to select a neighbour
  return np.random.choice(neighbors,size=1, p=prob)[0]

Now we have two hyperparameters to play around (p and q).


*   The value of q should lie between 1 and p. This is because the prob of visiting a node that is neighbour to current and previous should be maximum. so p and q are atleast greater than 1.
*   The probability of going back to the movie which is already watched should be minimum
* Hence 1 < q < p



In [64]:
#Check whether the function is working fine
print(next_movie(movies_graph, 1473, 2899, 5,4))

2100


In [41]:
print(movies_graph[1473][2899])

{'weight': 10.42263842504811}


# Create a Random Walk Across Graph


random_walk generates the Random Walks for all the nodes num_walks times

*   Essentaily the Time Complexity would be num_walks * num_nodes * num_steps * average_degrees

*   10 * 5000 * 15 * 250 ~ 187.5Million

So we precompute the prob vectors to minimize the time

In [78]:
def random_walk(graph, num_walks, num_steps, p, q):
  walks = []
  nodes = list(graph.nodes())

  for walk_iter in range(num_walks):
    random.shuffle(nodes)
    for node in tqdm(nodes, desc = f"Random Walk Iteration {walk_iter + 1} of {num_walks}"):
      walk = [node]

      #Randomly walk Num_steps by calling the next_movie Function
      while len(walk) < num_steps:
        current = walk[-1]
        previous = walk[-2] if len(walk) > 1 else None
        next = next_movie(graph, previous, current, p, q)
        walk.append(next)

      walks.append(walk)

  return walks

In [80]:
print(random_walk(movies_graph, 1,4, 3, 2))

Random Walk Iteration 1 of 1: 100%|██████████| 9724/9724 [06:28<00:00, 25.03it/s]

[[5009, np.int64(60161), np.int64(4267), np.int64(1390)], [8582, np.int64(1648), np.int64(2867), np.int64(4643)], [96488, np.int64(55080), np.int64(44191), np.int64(37477)], [640, np.int64(125), np.int64(40583), np.int64(180045)], [7786, np.int64(184721), np.int64(164179), np.int64(5297)], [619, np.int64(95067), np.int64(146688), np.int64(86298)], [636, np.int64(293), np.int64(55995), np.int64(109317)], [91671, np.int64(47952), np.int64(158254), np.int64(100507)], [8844, np.int64(4963), np.int64(40870), np.int64(541)], [219, np.int64(5), np.int64(152970), np.int64(27873)], [7940, np.int64(6286), np.int64(866), np.int64(2916)], [127180, np.int64(7151), np.int64(7766), np.int64(8784)], [470, np.int64(2709), np.int64(2399), np.int64(2084)], [3555, np.int64(6548), np.int64(3869), np.int64(2539)], [93297, np.int64(90469), np.int64(101112), np.int64(90531)], [36537, np.int64(7944), np.int64(6440), np.int64(2141)], [55294, np.int64(2942), np.int64(1837), np.int64(21)], [1333, np.int64(3981), 

In [81]:
import multiprocessing
print(multiprocessing.cpu_count())

2


In [ ]:
pip install node2vec

The project is executed on Google Colab (Free version), which provides up to 2 CPU cores. Therefore, even if a higher number of workers is specified, Colab automatically limits it to the available maximum.

In [ ]:
from node2vec import Node2Vec

node2vec = Node2Vec(movies_graph, dimensions=64, walk_length=30, num_walks=200, p=2, q=1.5, workers=4)